In [10]:
import os
import boto3

R2_ENDPOINT = "https://pub-c4575799aae84c49be5a8f49001724cd.r2.dev"
BUCKET = "gradutationgallery"

s3 = boto3.client(
    "s3",
    endpoint_url="https://282cef2a2c0a23e66a4773b8d6dee38f.r2.cloudflarestorage.com",
    aws_access_key_id="2cbec7a509491085f8e235f010c2712e",
    aws_secret_access_key="07f874b2afdb264a6dd764a48d4301aea3ea6ef6e0108f70e3f5e1f7d96a4ca8",
)

FOLDER = "data/images2"

for filename in os.listdir(FOLDER):
    path = os.path.join(FOLDER, filename)
    if not os.path.isfile(path):
        continue

    s3.upload_file(
        path,
        BUCKET,
        filename,
        ExtraArgs={"ContentType": "image/jpeg"}
    )

    print("uploaded:", filename)

uploaded: 7759656518142.mp4
uploaded: 7759658607215.mp4
uploaded: EHE07095.jpg
uploaded: EHE07096.jpg
uploaded: EHE07097.jpg
uploaded: EHE07098.jpg
uploaded: EHE07099.jpg
uploaded: EHE07100.jpg
uploaded: EHE07101.jpg
uploaded: EHE07102.jpg
uploaded: EHE07103.jpg
uploaded: EHE07104.jpg
uploaded: EHE07106.jpg
uploaded: EHE07107.jpg
uploaded: EHE07108.jpg
uploaded: EHE07109.jpg
uploaded: EHE07110.jpg
uploaded: EHE07111.jpg
uploaded: EHE07112.jpg
uploaded: EHE07113.jpg
uploaded: EHE07114.jpg
uploaded: EHE07115.jpg
uploaded: EHE07116.jpg
uploaded: EHE07117.jpg
uploaded: EHE07118.jpg
uploaded: EHE07119.jpg
uploaded: EHE07120.jpg
uploaded: EHE07121.jpg
uploaded: EHE07122.jpg
uploaded: EHE07123.jpg
uploaded: EHE07124.jpg
uploaded: EHE07125.jpg
uploaded: EHE07126.jpg
uploaded: EHE07127.jpg
uploaded: EHE07128.jpg
uploaded: EHE07129.jpg
uploaded: EHE07130.jpg
uploaded: EHE07131.jpg
uploaded: EHE07132.jpg
uploaded: EHE07133.jpg
uploaded: EHE07135.jpg
uploaded: EHE07136.jpg
uploaded: EHE07137.jpg
u

In [9]:
import os
import json
import uuid
import tkinter as tk
from PIL import Image, ImageTk
import tkinter.font as tkFont

# ===== CONFIG =====
IMAGES_DIR = r"C:\HoangTu\Programing\Gradutation\data\images2"
OUT_DIR = r"C:\HoangTu\Programing\Gradutation\data"

IMAGES_JSON = os.path.join(OUT_DIR, "images_data.json")
OBJECTS_JSON = os.path.join(OUT_DIR, "objects_meta.json")

os.makedirs(OUT_DIR, exist_ok=True)

# ===== LOAD =====
if os.path.exists(IMAGES_JSON):
    with open(IMAGES_JSON, "r") as f:
        images_data = json.load(f)
else:
    images_data = []

if os.path.exists(OBJECTS_JSON):
    with open(OBJECTS_JSON, "r", encoding="utf-8") as f:
        objects_meta = json.load(f)
else:
    objects_meta = {}

# name -> id
name_to_id = {v["name"]: k for k, v in objects_meta.items()}

# ===== IMAGE LIST =====
image_files = [
    os.path.join(IMAGES_DIR, f)
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# map image_path -> index trong images_data
image_map = {item["image_path"]: i for i, item in enumerate(images_data)}

index = 0

# ===== UI =====
root = tk.Tk()
root.title("Label Objects")

img_label = tk.Label(root)
img_label.pack()

font_style = tkFont.Font(family="Arial", size=18)

entry = tk.Entry(
    root,
    width=80,
    font=font_style,
    relief="solid",
    bd=2,
    justify="left"
)
entry.pack(ipady=10, padx=10, pady=10)

info_label = tk.Label(root, text="")
info_label.pack()

def load_image():
    global index

    if index < 0:
        index = 0
    if index >= len(image_files):
        index = len(image_files) - 1

    path = image_files[index]
    img = Image.open(path).convert("RGB")
    img.thumbnail((800, 600))

    tk_img = ImageTk.PhotoImage(img)
    img_label.config(image=tk_img)
    img_label.image = tk_img

    path_norm = path.replace("\\", "/")

    # load existing labels nếu có
    if path_norm in image_map:
        subjects = images_data[image_map[path_norm]]["subjects"]
        names = [objects_meta[s]["name"] for s in subjects if s in objects_meta]
        entry.delete(0, tk.END)
        entry.insert(0, ", ".join(names))
    else:
        entry.delete(0, tk.END)

    info_label.config(text=f"{index+1}/{len(image_files)}")

def save_current():
    path = image_files[index].replace("\\", "/")

    text = entry.get().strip()
    if not text:
        objs = []
    else:
        names = [x.strip() for x in text.split(",") if x.strip()]
        objs = []

        for name in names:
            if name in name_to_id:
                obj_id = name_to_id[name]
            else:
                obj_id = str(uuid.uuid4())
                name_to_id[name] = obj_id
                objects_meta[obj_id] = {
                    "id": obj_id,
                    "name": name
                }
            objs.append(obj_id)

    if path in image_map:
        images_data[image_map[path]]["subjects"] = objs
    else:
        images_data.append({
            "image_path": path,
            "subjects": objs
        })
        image_map[path] = len(images_data) - 1

    # save ngay
    with open(IMAGES_JSON, "w") as f:
        json.dump(images_data, f, indent=2)

    with open(OBJECTS_JSON, "w") as f:
        json.dump(objects_meta, f, indent=2)

def next_image(event=None):
    global index
    save_current()
    index += 1
    load_image()

def prev_image(event=None):
    global index
    save_current()
    index -= 1
    load_image()

# ===== BIND =====
root.bind("<Return>", next_image)
root.bind("<Right>", next_image)
root.bind("<Left>", prev_image)

btn_frame = tk.Frame(root)
btn_frame.pack()

btn_prev = tk.Button(btn_frame, text="Previous", command=prev_image)
btn_prev.pack(side="left", padx=10)

btn_next = tk.Button(btn_frame, text="Next", command=next_image)
btn_next.pack(side="left", padx=10)

# ===== START =====
load_image()
root.mainloop()